In [4]:
import xgboost as xgb
from sklearn.metrics import confusion_matrix,f1_score
from sklearn.metrics import accuracy_score,recall_score
from sklearn.model_selection import train_test_split
import pickle as pk
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import shap
from sklearn.preprocessing import StandardScaler

In [5]:
df=pd.read_csv("synthetic_subsidy_cylinders.csv")
df.head()



,Age,Gender,Marital_Status,Household_Size,Governorate,Salary,Degree_Level,Employment_Status,Primary_Income_Source,Has_Other_Social_Benefits,...,Average_Fuel_Consumption_L,Fuel_Deviation_L,Fuel_Deviation_Ratio,Previous_Subsidy_Received,Previous_Subsidy_Amount,Late_or_Missed_Renewals,Applications_Last_12_Months,Fraud,Eligible,ID
0,22.0,Male,Single,1,Muscat,96.92,Bachelor,Employed,Private,0,...,304.175,176.98,2.391,1,24.13,2,4,1,1,11597060
1,47.0,Male,Married,9,Sohar,144.06,Diploma,Employed,Government,0,...,104.860,-15.14,0.874,1,24.42,0,0,0,1,18838583
2,38.0,Female,Single,3,Dhofar,82.22,Diploma,Unemployed,Government,1,...,136.130,8.93,1.070,1,27.05,0,0,0,1,15247173
3,18.0,Male,Single,8,Sur,80.00,Diploma,Student,Government,0,...,203.480,14.48,1.077,0,5.00,1,1,0,1,14571592
4,28.0,Female,Single,3,Sohar,80.00,Diploma,Employed,Government,1,...,128.210,2.21,1.018,1,36.66,1,3,0,1,12044003


In [6]:
#to predicted eligibity 
LB = LabelEncoder()
for col in df:
    if df[col].dtype == "object":
        df[col]=LB.fit_transform(df[col])
dfe=df.drop(columns=["Fraud",'ID'])

X1=dfe.iloc[:,:-1]
y1=dfe.iloc[:,-1]
xtrain,xtest,ytrain,ytest=train_test_split(X1,y1,train_size=0.8,random_state=42)
xge=xgb.XGBClassifier()
xge.fit(xtrain,ytrain)
ype=xge.predict(xtest)

explainer = shap.TreeExplainer(xge)


shap_values = explainer.shap_values(xtest)
dfee= dfe.drop(columns=["Eligible"])
explaining=pd.DataFrame(columns=dfee.columns ,data=shap_values)
print(explaining)
print(shap_values)
print(confusion_matrix(ytest,ype))
print(f1_score(ytest,ype))
print(accuracy_score(ytest,ype))
print(recall_score(ytest,ype))



          Age  Gender  Marital_Status  Household_Size  Governorate    Salary  \
0   -0.589178     0.0       -0.073798        0.054429    -0.076544 -3.245175   
1   -1.227146     0.0       -0.235534       -0.038988     0.244346 -4.262181   
2    0.574714     0.0        0.093581       -0.038988     0.124076  3.292732   
3   -0.430375     0.0        0.632281        0.054429    -0.087733 -2.456080   
4   -0.270246     0.0        0.189384        0.054429    -0.074773  3.663790   
..        ...     ...             ...             ...          ...       ...   
155 -0.555881     0.0       -0.056226        0.054429    -0.076544 -0.429355   
156 -0.818981     0.0        0.439170       -0.038988     0.128007 -4.022184   
157 -1.240234     0.0       -0.238491       -0.038988    -0.098516 -3.942134   
158 -0.550788     0.0       -0.082137       -0.038988    -0.087081 -3.855043   
159 -0.292219     0.0        0.189473       -0.038988     0.156254 -3.306059   

     Degree_Level  Employment_Status  P

In [7]:
with open("elmodel.pk","wb") as file:
    pk.dump(xge,file)

In [ ]:
#print(np.random.default_rng().integers(10000000,19999999))
#df["ID"] = np.random.default_rng().integers(10000000,19999999,size=len(df))
df.to_csv("synthetic_subsidy_cylinders1.csv", index=False)